In [ ]:
import os, sys
REPO = os.environ.get("MTAH_REPO") or next((p for p in ["/kaggle/input/hackaton-barcellona-code", "/kaggle/working/Hackaton_Barcellona", os.path.abspath("..")] if os.path.isdir(os.path.join(p, "pipeline"))), None)
assert REPO, "Imposta MTAH_REPO alla cartella della repo"
sys.path[:0] = [REPO, os.path.join(REPO, "pipeline"), os.path.join(REPO, "starter"), os.path.join(REPO, "evaluation")]

# B2 · Regola "tocco + conferma del controllo" sulle 7 partite DFL

Il metodo 4 (`dfl_method4.py`) passa il possesso a **ogni** tocco: sulle 7 partite DFL simulate fa
spellF1 0,543 contro 0,421 della baseline. Ma la challenge cambia il possesso solo quando l'altra
squadra ottiene il **controllo** della palla. Questo notebook misura una regola che parte dagli
stessi tocchi e chiede una conferma del controllo prima di cambiare il possesso.

Tutta la logica è in `pipeline/touch_control.py`; qui si eseguono i passi e si leggono i risultati.
Le partite DFL sono altre partite (CC BY 4.0): tarare qui è permesso. Si stampano solo aggregati.

## 1. Tocco e controllo secondo le regole

Da `CHALLENGE.md`:

- il possesso cambia quando un giocatore dell'altra squadra ottiene il **possesso controllato**;
- **non** cambiano il possesso: un tocco senza controllo, una deviazione che non intercetta il
  passaggio, un duello aereo vinto senza controllo successivo;
- con la palla contesa o vagante il possesso **resta all'ultima squadra** che l'aveva.

Un *tocco* è un evento fisico (la palla cambia bruscamente velocità vicino a un giocatore), il
*controllo* è uno stato (la squadra dispone della palla). Il metodo 4 confonde i due: una
deviazione avversaria in mezzo a un passaggio spezza un possesso vero in tre, e spellF1 lo punisce
due volte (un possesso vero non accoppiato, due possessi falsi).

Cosa dice la letteratura (valori usati per la griglia):

- **Link & Hoernig 2017** (PLOS ONE, TRACAB 25 Hz): il tocco è un massimo locale dell'accelerazione
  della palla ≥ 4 m/s² con un giocatore vicino; il controllo si distingue dalla semplice vicinanza
  con durata, velocità/accelerazione della palla, distanza media e numero di avversari tra 0,5 e
  5 m. Fine del possesso: la palla prevista a 1 s esce dalla zona d'azione per 1 s. Accordo tra
  due osservatori umani sul controllo: kappa 0,72, un tetto realistico.
- **Vidal-Codina et al. 2022**: zona di possesso 0,5 m su tracking buono, **1 m** su tracking meno
  preciso; zona di duello 1 m con almeno 2 avversari; il controllo diventa guadagno di possesso
  solo con un cambio di direzione o di velocità della palla (cioè un tocco).
- **Borghesi et al. 2023** (riimplementazione di Morra): soglie 1,09 m e 1,19 m, velocità della
  palla 8,6 m/s; su tracking rumoroso zona di controllo **1-1,5 m**.
- **PathCRF (Kim et al. 2026**, sullo stesso dataset DFL): vincolare le transizioni impossibili
  riduce i cambi spuri; le transizioni dovrebbero dipendere dal contesto (pressione). Qui il
  vincolo è la conferma; la dipendenza dalla pressione resta un passo successivo (decoder A1).

La nostra soglia del tocco `T` è un salto di velocità in m/s su ±2 fotogrammi (0,16 s), quindi
`T` m/s ≈ 6,25·`T` m/s²: la griglia `T ∈ {1, 2, 3}` corrisponde a circa 6, 12,5 e 19 m/s².
Il valore di Link (≈ `T` 0,64) è pensato per dati puliti; col nostro rumore serve una soglia più
alta. `R ∈ {1; 1,5}` m viene da Vidal-Codina e Borghesi. La palla è levigata con una media mobile
di 5 fotogrammi (Vidal-Codina usa Savitzky-Golay, polinomio 2, finestra 7: un'alternativa da
provare).

## 2. La regola implementata

Dentro ogni tratto di palla viva (palla morta → `D`):

1. **Candidati:** i tocchi di `dfl_control.touches` (stessa soglia e stesso filtro del metodo 4:
   nessun tocco cercato dove la palla è quella di Metrica). Squadra del tocco = squadra del
   giocatore più vicino alla palla.
2. Un tocco della squadra X diversa da quella in possesso è **confermato** se nella finestra
   (tocco, tocco + `T1` s], chiusa prima al primo tocco dell'altra squadra e alla fine del tratto
   vivo, succede almeno una di queste cose:
   - **(a)** X tocca di nuovo la palla (secondo tocco o passaggio a un compagno);
   - **(b)** un giocatore di X resta entro `R` m dalla palla, più vicino di ogni avversario, per
     `N` fotogrammi di fila (conduzione);
   - **(c)** la palla rallenta sotto 5 m/s entro 2 m da un giocatore di X, più vicino di ogni
     avversario, per 3 fotogrammi (ricezione);
   - **(d)** X aveva già la palla entro `R` m per `N` fotogrammi fino al tocco (dribbling o calcio
     piazzato il cui primo contatto non si è visto): il cambio si data all'inizio di quel tratto.
3. Tocco **non confermato** (deviazione, rimpallo): il possesso resta all'ultima squadra.
4. **Riserva** (opzionale, `fallback_n`): dove il tocco non si vede (palla interpolata o di
   Metrica), un giocatore di X entro `R` m e più vicino di ogni avversario per `fallback_n`
   fotogrammi passa la palla a X, datata all'inizio del tratto.
5. **Ripresa dopo `D`:** la squadra è ignota; il primo evento confermato la decide e viene scritto
   all'indietro fino al primo fotogramma vivo (alla ripresa c'è sempre un possessore chiaro).

### Perché il cambio si data al primo tocco

La conferma arriva dopo, anche di 1-1,5 s. Se il cambio fosse datato alla conferma, ogni confine
sarebbe in ritardo di quella quantità: su spellF1 1 s di ritardo su un cambio costa circa 0,35,
perché l'IoU dei due possessi adiacenti scende sotto le soglie 0,3-0,7. La regola lavora a
posteriori sull'intera partita, quindi può guardare avanti per confermare e scrivere il cambio
all'indietro, al tocco che l'ha iniziato.

### Uscita morbida `p_home`

Per il decoder A1: `p_home` = 0,5 a palla morta; a palla viva 0,5 ± c/2 con c la confidenza
dell'ultimo evento della squadra in possesso, che decresce col tempo (8 s) e più in fretta mentre
la palla non è la nostra (interpolata o di Metrica, 2 s), e si riduce a ogni tocco avversario non
confermato. Il lato di `p_home` coincide sempre con `labels`.

## 3. Controllo rapido su scenari sintetici

Passaggio intercettato e controllato, deviazione senza controllo, secondo tocco, vuoto di palla,
palla morta e ripresa, una pipeline completa su due partite finte. Non servono dati veri.

In [ ]:
import test_touch_control as ttc
for name in sorted(n for n in dir(ttc) if n.startswith("test_")):
    getattr(ttc, name)()
    print("ok", name)

## 4. Dati e condizioni realistiche

Le partite convertite si trovano con `dfl_paths.converted_dir()` (`$DFL_CONVERTED`, la cartella
locale, oppure il primo dataset Kaggle con file `DFL-MAT-*.npz`). Le uscite vanno in
`dfl_paths.out_dir()` (`$DFL_OUT`, `/kaggle/working/dfl_outputs` su Kaggle).

Condizioni di `dfl_method4.py`, nella catena corretta di CONTESTO.md §6 (nessun vuoto di Metrica a
gioco vivo):

- **palla ferma** dove manca la palla tipo Metrica (90% delle interruzioni oltre 1 s), oppure da
  `<match>_deadball.npz` di B1 con `DEAD = "b1"` / `"auto"`;
- **nostra palla** = palla vera + 0,03 m, presente/assente a tratti con le durate del nostro
  rilevatore (`ball_runs.npz`, solo aggregati, via `$DFL_BALL_RUNS`; se manca si usano durate
  sintetiche e lo si dice); vuoti ≤ 2 s interpolati, Metrica (media su 12 fotogrammi) nel resto;
- **giocatori** + 0,2 m, più l'1% di squadra sbagliata al portatore; semi fissi.

In [ ]:
import numpy as np, pandas as pd
import dfl_paths
import touch_control as tc

SEEDS = (0,)          # (0, 1) per due semi: il doppio del tempo
DEAD = "metrica"      # "metrica" | "b1" | "auto"
QUICK = False         # True: griglia ridotta
GRID = tc.QUICK_GRID if QUICK else tc.GRID
paths = dfl_paths.match_paths()
print(len(paths), "partite in", dfl_paths.converted_dir())
print("griglia:", GRID, "->", len(tc.grid_configs(GRID)), "configurazioni")

## 5. Esecuzione: baseline, metodo 4, tocco + controllo

Per ogni partita e seme: baseline ±50 (palla di Metrica), metodo 4 con `T ∈ {1, 2, 3}`, tocco +
controllo su tutta la griglia `T × T1 × R × N × fallback`. Poi **leave-one-match-out**: per ogni
partita i parametri si scelgono sulle altre 6 (spellF1 medio) e si misurano sulla 7ª. Vale anche
per la soglia del metodo 4, così il confronto è alla pari. Qualche minuto.

In [ ]:
df, kept, rl_kind = tc.evaluate_all(paths, SEEDS, GRID, DEAD, keep={dfl_paths.match_name(p) for p in paths})
held, chosen = tc.lomo(df)
print("durate dei vuoti della palla:", rl_kind)

## 6. Risultati (leave-one-match-out)

Come leggere la tabella:

- `spell_f1` è la metrica della classifica, media sulle 7 partite tenute fuori; `spell_min` /
  `spell_max` danno la dispersione tra partite. La banda di rumore di spellF1 su una partita è
  circa 0,04: differenze più piccole non sono un risultato.
- `macro_f1` è il primo spareggio; si muove poco, perché quasi tutti i fotogrammi stanno dentro un
  possesso.
- `n_pred` contro `n_gt` (possessi per partita): se `n_pred` è molto sopra, la regola
  **sovra-segmenta** (cambi falsi, tipico del metodo 4); se è sotto, **sotto-segmenta** (cambi
  persi, il rischio di una conferma troppo severa).

In [ ]:
pd.set_option("display.width", 200)
tc.summary(held)[["spell_f1", "spell_min", "spell_max", "macro_f1", "n_pred", "n_gt"]]

In [ ]:
# quali parametri sceglie il leave-one-match-out (quante volte su 7)
pd.Series(chosen).groupby(level=0).value_counts().rename("volte").to_frame()

In [ ]:
# tocco + controllo, migliori configurazioni su tutte le partite (in-sample: solo indicativo)
tc.top_configs(df, k=12)

## 7. Analisi degli errori

Sui soli cambi di possesso H↔A (non le transizioni verso o da `D`), accoppiati con la verità entro
±1 s (25 fotogrammi), medie per partita:

- `missed`: cambi veri senza un cambio predetto vicino;
- `spurious`: cambi predetti senza un cambio vero vicino;
- `lat_median`: ritardo mediano (predetto − vero, fotogrammi; positivo = in ritardo) dei cambi
  accoppiati; `abs_lat_median` il suo valore assoluto.

Il metodo 4 dovrebbe avere molti `spurious` (ogni deviazione); la conferma li deve ridurre senza
far crescere troppo `missed`. Una latenza vicina a 0 conferma che datare al primo tocco funziona.

In [ ]:
s = tc.summary(held)
s["missed_%"] = (100 * s.missed / s.n_changes_gt).round(1)
s["spurious_per_true"] = (s.spurious / s.n_changes_gt).round(2)
s[["n_changes_gt", "missed", "missed_%", "spurious", "spurious_per_true", "lat_median", "abs_lat_median"]]

In [ ]:
# tocchi non confermati per partita (deviazioni / rimpalli scartati), con la config scelta per partita
tcr = df[df.method == "tocco + controllo"]
sel = tcr.match.map({m: chosen[("tocco + controllo", m)] for m in tcr.match.unique()})
x = tcr[tcr.cfg == sel].groupby("match").n_unconfirmed.mean()
print("tocchi non confermati per partita: media {:.0f}, min {:.0f}, max {:.0f}".format(x.mean(), x.min(), x.max()))

## 8. Un tratto DFL: verità, metodo 4, tocco + controllo

Un minuto di una partita DFL (il tratto con più cambi veri dopo il 10° minuto), con i parametri
scelti senza quella partita. Triangoli in alto: tocchi rilevati; **pieni** = cambio confermato,
**vuoti** = tocco avversario non confermato (dove il metodo 4 cambia e la nuova regola no),
piccoli grigi = tocchi della squadra già in possesso.

In [ ]:
import matplotlib.pyplot as plt
match = sorted(kept)[0]
cfg = tc.parse_cfg(chosen[("tocco + controllo", match)])
fig, ax = plt.subplots(figsize=(13, 3.2))
tc.plot_segment(kept[match], cfg, seconds=60, ax=ax)
plt.tight_layout(); plt.show()

## 9. Uscite per il decoder A1

Per ogni partita `<match>_rule.npz` in `dfl_paths.out_dir()` con `labels` (H/A/D), `p_home`
(float32) e `variant`, prodotti con i parametri scelti **senza** quella partita (nessuna fuga di
taratura verso A1).

In [ ]:
written = tc.save_outputs(kept, chosen, SEEDS[0])
z = np.load(written[0])
print(len(written), "file in", dfl_paths.out_dir())
print("chiavi:", list(z.keys()), "| variant:", str(z["variant"]))

## 10. Note e passi successivi

- Il confronto onesto è quello leave-one-match-out; la tabella in-sample serve solo a vedere quanto
  è piatta la griglia.
- Da CLI: `python pipeline/touch_control.py [--seeds 0 1] [--quick] [--dead auto] [--gap-touches]`.
- Da provare: tocchi attraverso i vuoti (`--gap-touches`, `dfl_control.touches_across_gaps`),
  conferma dipendente dalla pressione (numero di avversari tra 0,5 e 5 m, Link & Hoernig;
  PathCRF), levigatura Savitzky-Golay della palla, `p_home` come emissione del decoder A1.